# LlamaIndex 快速開始教程

本教程將幫助你快速上手 LlamaIndex，構建第一個 RAG（檢索增強生成）應用。

## 📚 學習目標

- 理解 LlamaIndex 的核心概念
- 安裝和配置 LlamaIndex
- 創建第一個索引
- 執行查詢並獲取響應
- 理解基本的 RAG 工作流程

## 1. 環境設置和安裝

首先安裝必要的套件：

In [ ]:
# 安裝 LlamaIndex 核心套件
!pip install llama-index llama-index-core llama-index-llms-openai llama-index-embeddings-openai -q

## 2. 配置 API Key

LlamaIndex 支持多種 LLM 提供商，這裡我們使用 OpenAI：

In [ ]:
import os
from dotenv import load_dotenv

# 加載環境變數
load_dotenv()

# 設置 OpenAI API Key
# 方法 1: 從 .env 文件加載
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 方法 2: 直接設置（僅用於測試）
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("API Key 已設置！")

## 3. 核心概念介紹

### LlamaIndex 的三個核心組件：

1. **Documents（文檔）**: 你的原始數據
2. **Index（索引）**: 組織和存儲數據的結構
3. **Query Engine（查詢引擎）**: 用於查詢索引的接口

### 工作流程：
```
數據 → 文檔 → 索引 → 查詢引擎 → 響應
```

## 4. 創建示例數據

讓我們創建一些示例文本數據：

In [ ]:
import os

# 創建數據目錄
os.makedirs("data", exist_ok=True)

# 創建示例文件
sample_texts = {
    "ai_intro.txt": """人工智慧（AI）是計算機科學的一個分支，致力於創建能夠執行通常需要人類智慧的任務的系統。
    AI 包括機器學習、深度學習、自然語言處理等多個領域。近年來，大型語言模型（LLM）如 GPT、Claude 等的出現，
    極大地推動了 AI 技術的發展。""",
    
    "machine_learning.txt": """機器學習是人工智慧的一個子領域，專注於開發能夠從數據中學習的算法。
    主要分為監督學習、非監督學習和強化學習三大類。監督學習使用標記數據進行訓練，
    非監督學習從未標記數據中發現模式，強化學習通過與環境互動來學習最優策略。""",
    
    "llm_applications.txt": """大型語言模型（LLM）的應用非常廣泛，包括：
    1. 對話系統和聊天機器人
    2. 文本生成和內容創作
    3. 代碼生成和輔助編程
    4. 文檔問答和知識檢索
    5. 翻譯和文本摘要
    6. 情感分析和文本分類
    RAG（檢索增強生成）技術結合了檢索和生成，能夠提供更準確、更有依據的回答。"""
}

# 寫入文件
for filename, content in sample_texts.items():
    with open(f"data/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("示例數據已創建！")
print(f"文件列表: {list(sample_texts.keys())}")

## 5. 加載文檔

使用 `SimpleDirectoryReader` 加載數據目錄中的所有文件：

In [ ]:
from llama_index.core import SimpleDirectoryReader

# 加載文檔
documents = SimpleDirectoryReader("data").load_data()

print(f"加載了 {len(documents)} 個文檔")
print(f"\n第一個文檔的內容預覽：\n{documents[0].text[:200]}...")

## 6. 創建向量索引

`VectorStoreIndex` 是最常用的索引類型，它將文檔轉換為向量嵌入：

In [ ]:
from llama_index.core import VectorStoreIndex

# 創建索引（這一步會調用 OpenAI API 生成嵌入）
print("正在創建索引...")
index = VectorStoreIndex.from_documents(documents)
print("索引創建完成！")

## 7. 創建查詢引擎

查詢引擎用於對索引進行查詢：

In [ ]:
# 創建查詢引擎
query_engine = index.as_query_engine()

print("查詢引擎已準備就緒！")

## 8. 執行查詢

現在我們可以向系統提問了：

In [ ]:
# 查詢 1: 關於 AI 的基本問題
response = query_engine.query("什麼是人工智慧？")
print("問題: 什麼是人工智慧？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 2: 關於機器學習的問題
response = query_engine.query("機器學習有哪些主要類型？")
print("問題: 機器學習有哪些主要類型？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 3: 關於 LLM 應用的問題
response = query_engine.query("LLM 可以應用在哪些場景？")
print("問題: LLM 可以應用在哪些場景？")
print(f"回答: {response}")
print("\n" + "="*80 + "\n")

In [ ]:
# 查詢 4: 關於 RAG 的問題
response = query_engine.query("什麼是 RAG 技術？")
print("問題: 什麼是 RAG 技術？")
print(f"回答: {response}")

## 9. 查看檢索到的原始文檔

我們可以看到 LlamaIndex 從哪些文檔中檢索了信息：

In [ ]:
# 創建帶有詳細信息的查詢引擎
query_engine_detailed = index.as_query_engine(response_mode="tree_summarize")

response = query_engine_detailed.query("解釋一下 RAG 技術")

print("回答:", response)
print("\n" + "="*80)
print("檢索到的源文檔數量:", len(response.source_nodes))
print("\n原始文檔內容:")
for i, node in enumerate(response.source_nodes, 1):
    print(f"\n文檔 {i}:")
    print(f"相似度分數: {node.score:.4f}")
    print(f"內容: {node.text[:200]}...")

## 10. 自定義查詢參數

我們可以調整查詢引擎的參數：

In [ ]:
# 自定義查詢引擎：返回更多相關文檔
query_engine_custom = index.as_query_engine(
    similarity_top_k=5,  # 返回最相似的 5 個文檔塊
    response_mode="compact"  # 使用緊湊模式
)

response = query_engine_custom.query("總結一下關於人工智慧和機器學習的信息")
print(f"回答: {response}")

## 11. 保存和加載索引

我們可以保存索引以避免重複創建：

In [ ]:
# 保存索引
index.storage_context.persist(persist_dir="./storage")
print("索引已保存到 ./storage")

In [ ]:
# 加載索引
from llama_index.core import StorageContext, load_index_from_storage

# 加載存儲的索引
storage_context = StorageContext.from_defaults(persist_dir="./storage")
loaded_index = load_index_from_storage(storage_context)

# 使用加載的索引
loaded_query_engine = loaded_index.as_query_engine()
response = loaded_query_engine.query("什麼是人工智慧？")
print(f"使用加載的索引查詢結果: {response}")

## 12. 使用其他 LLM（可選）

LlamaIndex 支持多種 LLM 提供商：

In [ ]:
# 示例：使用不同的 LLM
# 需要先安裝: pip install llama-index-llms-gemini llama-index-llms-ollama

from llama_index.core import Settings

# 方法1: 使用 Google Gemini
# from llama_index.llms.gemini import Gemini
# Settings.llm = Gemini(model="models/gemini-pro", api_key="your-api-key")

# 方法2: 使用本地 Ollama (完全免費!)
# from llama_index.llms.ollama import Ollama
# Settings.llm = Ollama(model="llama3", base_url="http://localhost:11434")

# 方法3: 使用 Azure OpenAI
# from llama_index.llms.azure_openai import AzureOpenAI
# Settings.llm = AzureOpenAI(
#     model="gpt-4",
#     deployment_name="your-deployment",
#     api_key="your-key",
#     azure_endpoint="https://your-endpoint.openai.azure.com"
# )

print("💡 提示：取消註釋上面的程式碼來使用不同的 LLM")

## 📝 總結

在本教程中，我們學習了：

1. ✅ LlamaIndex 的安裝和配置
2. ✅ 核心概念：Documents、Index、Query Engine
3. ✅ 加載文檔數據
4. ✅ 創建向量索引
5. ✅ 執行查詢並獲取響應
6. ✅ 查看檢索的源文檔
7. ✅ 自定義查詢參數
8. ✅ 保存和加載索引
9. ✅ 支持多種 LLM

## 🎯 下一步

- 學習更多數據加載器：`1.數據加載與索引.ipynb`
- 探索不同的查詢引擎：`2.查詢引擎.ipynb`
- 構建聊天系統：`3.Chat_Engine聊天引擎.ipynb`

## 🔗 參考資源

- [LlamaIndex 官方文檔](https://docs.llamaindex.ai/)
- [LlamaIndex GitHub](https://github.com/run-llama/llama_index)
- [Discord 社區](https://discord.gg/dGcwcsnxhU)

## 13. 進階：使用不同的嵌入模型

嵌入模型決定了文本如何被向量化，選擇合適的模型很重要：

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding

# 使用更好的嵌入模型（推薦用於生產環境）
Settings.embed_model = OpenAIEmbedding(
    model="text-embedding-3-large",  # 更高維度，更好的語義理解
    dimensions=1024  # 可以指定維度（256-3072）
)

# 重新創建索引使用新的嵌入模型
index_with_better_embedding = VectorStoreIndex.from_documents(documents)

query_engine_better = index_with_better_embedding.as_query_engine()
response = query_engine_better.query("什麼是人工智慧？")
print(f"使用更好嵌入模型的回答: {response}")

## 14. 進階：流式響應

在生產環境中，流式響應可以提供更好的用戶體驗：

In [ ]:
# 創建流式查詢引擎
streaming_query_engine = index.as_query_engine(streaming=True)

# 執行流式查詢
print("問題: 解釋一下 LLM 的應用場景")
print("回答: ", end="")

streaming_response = streaming_query_engine.query("解釋一下 LLM 的應用場景")

# 逐字輸出（模擬打字效果）
for text in streaming_response.response_gen:
    print(text, end="", flush=True)

print("\n" + "="*80)

## 15. 進階：調試和評估

了解系統如何工作對於優化很重要：

In [ ]:
# 啟用詳細日誌
import logging
import sys

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

# 執行查詢並查看詳細日誌
query_engine_debug = index.as_query_engine()
response_debug = query_engine_debug.query("什麼是 RAG？")

print("\n" + "="*80)
print("回答:", response_debug)
print("\n來源節點數量:", len(response_debug.source_nodes))

# 查看每個檢索到的節點的相似度分數
for i, node in enumerate(response_debug.source_nodes, 1):
    print(f"\n節點 {i}:")
    print(f"  相似度分數: {node.score:.4f}")
    print(f"  文本長度: {len(node.text)} 字符")
    print(f"  文本預覽: {node.text[:100]}...")

## 16. 完整的實戰範例：構建一個簡單的文檔問答系統

讓我們整合所有學到的知識，構建一個完整的系統：

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, StorageContext, load_index_from_storage
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
import os

class DocumentQA:
    """簡單的文檔問答系統"""
    
    def __init__(self, data_dir="data", storage_dir="./storage", use_cache=True):
        self.data_dir = data_dir
        self.storage_dir = storage_dir
        self.use_cache = use_cache
        self.index = None
        
        # 配置模型
        Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)
        Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
        
    def build_index(self):
        """構建或加載索引"""
        # 嘗試從緩存加載
        if self.use_cache and os.path.exists(self.storage_dir):
            print("📂 從緩存加載索引...")
            storage_context = StorageContext.from_defaults(persist_dir=self.storage_dir)
            self.index = load_index_from_storage(storage_context)
            print("✅ 索引加載完成")
        else:
            print("🔨 構建新索引...")
            documents = SimpleDirectoryReader(self.data_dir).load_data()
            print(f"📄 已加載 {len(documents)} 個文檔")
            
            self.index = VectorStoreIndex.from_documents(documents)
            
            # 保存索引
            self.index.storage_context.persist(persist_dir=self.storage_dir)
            print("✅ 索引構建並保存完成")
    
    def query(self, question, similarity_top_k=3, streaming=False):
        """執行查詢"""
        if self.index is None:
            raise ValueError("請先構建索引：調用 build_index()")
        
        query_engine = self.index.as_query_engine(
            similarity_top_k=similarity_top_k,
            streaming=streaming
        )
        
        return query_engine.query(question)
    
    def chat(self):
        """互動式聊天"""
        if self.index is None:
            raise ValueError("請先構建索引：調用 build_index()")
        
        print("\\n💬 進入聊天模式（輸入 'exit' 退出）\\n")
        
        while True:
            question = input("你的問題: ")
            if question.lower() in ['exit', 'quit', '退出']:
                print("👋 再見！")
                break
            
            if not question.strip():
                continue
            
            try:
                response = self.query(question)
                print(f"\\n🤖 回答: {response}\\n")
                print("-" * 80)
            except Exception as e:
                print(f"❌ 錯誤: {e}")

# 使用示例
qa_system = DocumentQA(data_dir="data", storage_dir="./storage")
qa_system.build_index()

# 單次查詢
response = qa_system.query("什麼是機器學習？")
print(f"\\n回答: {response}\\n")

# 如果想進入聊天模式，取消下面的註釋
# qa_system.chat()

## 📝 本教程總結

恭喜！你已經完成了 LlamaIndex 快速開始教程。讓我們回顧一下學到的內容：

### ✅ 已掌握的知識

1. **基礎概念**:
   - Document（文檔）、Node（節點）、Index（索引）
   - Query Engine（查詢引擎）的工作原理
   - RAG（檢索增強生成）的基本流程

2. **核心技能**:
   - ✅ 安裝和配置 LlamaIndex
   - ✅ 加載和處理文檔
   - ✅ 創建向量索引
   - ✅ 執行查詢並獲取回答
   - ✅ 保存和加載索引
   - ✅ 查看檢索到的源文檔

3. **進階功能**:
   - ✅ 使用不同的 LLM（OpenAI、Gemini、Ollama）
   - ✅ 配置不同的嵌入模型
   - ✅ 自定義查詢參數
   - ✅ 實現流式響應
   - ✅ 調試和評估系統
   - ✅ 構建完整的文檔問答系統

### 🎯 下一步學習

1. **1.進階索引技術.ipynb**: 學習 Tree、Keyword、Graph 等多種索引
2. **2.數據加載與處理.ipynb**: 掌握 100+ 種數據源的加載方法
3. **3.查詢引擎深入.ipynb**: 深入理解查詢優化和檢索策略
4. **4.Chat_Engine聊天引擎.ipynb**: 構建對話系統
5. **5.Agents與工具使用.ipynb**: 學習 Agent 和工具整合

### 💡 實踐建議

1. **用自己的數據實驗**: 嘗試加載你自己的文檔（PDF、Word、網頁等）
2. **調整參數**: 修改 `chunk_size`、`similarity_top_k` 等參數觀察效果
3. **嘗試不同模型**: 對比 GPT-3.5、GPT-4、本地模型的效果差異
4. **優化成本**: 使用 `text-embedding-3-small` 和緩存來降低費用
5. **評估質量**: 記錄問題和答案，持續改進系統

### 🔗 有用的資源

- [官方文檔](https://docs.llamaindex.ai/)
- [GitHub 範例](https://github.com/run-llama/llama_index/tree/main/docs/examples)
- [Discord 社群](https://discord.gg/dGcwcsnxhU)
- [YouTube 教學](https://www.youtube.com/@LlamaIndex)

### 🚀 準備好了嗎？

繼續學習下一個教程，深入掌握 LlamaIndex 的強大功能！